<a id="pre"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Business Problem</p>

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">


- “For insurance companies, utilizing customer data is not just beneficial — it’s essential. Features such as age, gender, lifestyle habits, and health indicators play a pivotal role in evaluating risk, customizing coverage plans, and optimizing pricing strategies.”

<a id="pre"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Dataset Story</p>

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

📊 This dataset contains customer-level data used to analyze and predict healthcare expenses and insurance premiums. It includes demographic, behavioral, and regional features such as age, gender, BMI, number of children, and discount eligibility. The goal is to understand how these attributes influence annual medical expenses and insurance premiums, and to support decision-making in pricing and risk evaluation.

🔍 Feature Descriptions:

🧓 AGE: Age of the insured person.

🧔 GENDER: Gender of the individual (0 = female, 1 = male).

🧈 BMI (Body Mass Index): An indicator of body fat based on height and weight.

👶 CHILDREN: Number of dependents or children.

🏷️ DISCOUNT_ELIGIBILITY: Whether the individual is eligible for a health-related discount (e.g., based on smoking status or wellness behavior).

🌍 REGION: The geographic area of the insured (e.g., southeast, northwest, etc.).

💰 EXPENSES: Actual medical expenses incurred during the year.

📦 PREMIUM: The annual insurance premium paid by the customer.

🧮 RISK_SCORE (engineered feature): A custom score combining age, obesity status, and discount eligibility to estimate the individual’s health risk level.

In [ ]:
import pandas as pd

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import chardet

import csv
import itertools
import math

In [ ]:
data = pd.read_csv("C:\\Users\\SONY\\Downloads\\Copy of medical_insurance.csv")
data


<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">
• age: Age

 • gender: Gender

 • bmi: Body Mass Index

 • children: Number of children

 • discount_eligibility: Whether the person is eligible for a discount.

 • region: Geographic region

 • expenses: Medical expenses
 
 • premium: Insurance premium

In [ ]:
df=pd.DataFrame(data)
df


In [ ]:
df.describe(include="all")

In [ ]:
df.describe(include="number")

In [ ]:
df.describe(include="object")

In [ ]:
df.info()

In [ ]:
print(df.columns)

In [ ]:
def describe(df):
    
    
    variables = []   # Column names
    dtypes = []    # Data types of columns
    count = []      # Total number of entries in each column
    unique = []      # Number of unique values in each column
    missing = []    # Count of missing (NaN) values in each column
    
     # Loop through each column in the DataFrame
    for item in df.columns:
        variables.append(item)    # Append column name
        dtypes.append(df[item].dtype)  # Append column data type
        count.append(len(df[item]))   
        unique.append(len(df[item].unique()))
        missing.append(df[item].isna().sum())
        

    output = pd.DataFrame({
        'Variable': variables, 
        'Dtype': dtypes,
        'Count': count,
        'Unique': unique,
        'Missing value': missing
    })    
        
    return output

In [ ]:
describe(df)

In [ ]:
percent_missing = df.isnull().sum() * 100 / len(df)
missing_value_df = pd.DataFrame({'column_name': df.columns,
                                 'percent_missing': percent_missing})
percent = pd.DataFrame(missing_value_df)
percent

In [ ]:
# Display unique values for each column
for column in df.columns:
    print(f"Unique values in '{column}':")
    print(df[column].unique())
    print("-" * 40)

<a id="pre"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Data preparation</p>


In [ ]:
print(df.nunique())

In [ ]:
for column in df.columns:
    print(f"Value counts for '{column}':")
    print(df[column].value_counts())
    print("-" * 40)

In [ ]:
df.isnull().sum()

In [ ]:
# identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

# result
print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

In [ ]:


df[numeric_cols].describe().transpose()

<a id="pre"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Outlier Detection</p>

In [ ]:
sns.set_style('darkgrid')
colors = ['#0055ff', '#ff7000', '#23bf00', '#aa00ff', '#00ffaa', '#ff22aa', '#ffaa00', '#00aaff']
CustomPalette = sns.set_palette(sns.color_palette(colors))

OrderedCols = np.concatenate([df.select_dtypes(exclude='object').columns.values, 
                              df.select_dtypes(include='object').columns.values])

fig, ax = plt.subplots(2, 4, figsize=(15,7),dpi=100)

for i,col in enumerate(OrderedCols):
    x = i//4
    y = i%4
    if i<5:
        sns.boxplot(data=df, y=col, ax=ax[x,y],color=colors[i % len(colors)])
        ax[x,y].yaxis.label.set_size(15)
    else:
        sns.boxplot(data=df, x=col, y='expenses', ax=ax[x,y],color=colors[i % len(colors)])
        ax[x,y].xaxis.label.set_size(15)
        ax[x,y].yaxis.label.set_size(15)

plt.tight_layout()    
plt.show()

In [ ]:
# select numeric columns
#identify outliers
numerical_cols = ['age', 'bmi', 'children', 'expenses', 'premium'] 
# calculate the upper and lower range for each numeric column.
for col in numerical_cols:
    print(f"\n--- IQR Bounds for {col} ---") 
    Q1 = df[col].quantile(0.25) 
    Q3 = df[col].quantile(0.75) 
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR 
    upper_bound = Q3 + 1.5 * IQR 
    print(f"Q1 (25th percentile): {Q1}")
    print(f"Q3 (75th percentile): {Q3}") 
    print(f"IQR: {IQR}")
    print(f"Lower bound: {lower_bound}") 
    print(f"Upper bound: {upper_bound}") 
    # outliers 
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
     
    print(f"Number of outliers: {len(outliers)}")

 

In [ ]:
outliers_indexes = []
target = 'expenses'

for col in df.select_dtypes(include='object').columns:
    for cat in df[col].unique():
        df1 = df[df[col] == cat]
        q1 = df1[target].quantile(0.25)
        q3 = df1[target].quantile(0.75)
        iqr = q3-q1
        maximum = q3 + (1.5 * iqr)
        minimum = q1 - (1.5 * iqr)
        outlier_samples = df1[(df1[target] < minimum) | (df1[target] > maximum)]
        outliers_indexes.extend(outlier_samples.index.tolist())
        
        
for col in df.select_dtypes(exclude='object').columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3-q1
    maximum = q3 + (1.5 * iqr)
    minimum = q1 - (1.5 * iqr)
    outlier_samples = df[(df[col] < minimum) | (df[col] > maximum)]
    outliers_indexes.extend(outlier_samples.index.tolist())
    
outliers_indexes = list(set(outliers_indexes))
print('{} outliers were identified, whose indices are:\n\n{}'.format(len(outliers_indexes), outliers_indexes))

In [ ]:
df.loc[outliers_indexes]

In [ ]:
def grab_col_names(dataframe, cat_th=10, car_th=20):
    """

    Returns the names of categorical, numeric and categorical but cardinal variables in the data set.
    Note Categorical variables include categorical variables with numeric appearance.

    Parameters
    ------
        dataframe: dataframe
                Variable names of the dataframe to be taken
        cat_th: int, optional
                class threshold for numeric but categorical variables
        car_th: int, optinal
                class threshold for categorical but cardinal variables

    Returns
    ------
        cat_cols: list
                Categorical variable list
        num_cols: list
                Numeric variable list
        cat_but_car: list
                List of cardinal variables with categorical appearance

    Examples
    ------
        import seaborn as sns
        df = sns.load_dataset("iris")
        print(grab_col_names(df))


    Notes
    ------
        cat_cols + num_cols + cat_but_car = total number of variables
        num_but_cat is inside cat_cols.
        The sum of the 3 return lists equals the total number of variables: cat_cols + num_cols + cat_but_car = number of variables

    """

    # cat_cols, cat_but_car
    cat_cols = [col for col in dataframe.columns if dataframe[col].dtypes == "O"]

    num_but_cat = [col for col in dataframe.columns if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtypes != "O"]

    cat_but_car = [col for col in dataframe.columns if dataframe[col].nunique() > car_th and
                   dataframe[col].dtypes == "O"]

    cat_cols = cat_cols + num_but_cat

    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    num_cols = [col for col in dataframe.columns if dataframe[col].dtypes != "O"]

    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f'cat_cols: {len(cat_cols)}')
    print(f'num_cols: {len(num_cols)}')
    print(f'cat_but_car: {len(cat_but_car)}')
    print(f'num_but_cat: {len(num_but_cat)}')


    return cat_cols, num_cols, cat_but_car, num_but_cat

In [ ]:
cat_cols, num_cols, cat_but_car,  num_but_cat = grab_col_names(df)

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- The dataset contains 1338 rows and 8 features. Using the grab_col_names() function, the features are categorized as follows:

 • 4 categorical features: This includes both object-type columns (e.g., gender, region) and numeric 
 
 columns with low cardinality (e.g., children).

 • 4 numerical features: These are continuous variables such as age, bmi, and premium.

 • 0 cardinal categorical features: There are no object-type columns with high cardinality, so there’s no
 
  concern about high-dimensionality from categorical variables.

 • 1 numerical but categorical feature: One numeric column (likely children) has a small number of unique

  values and is treated as categorical for modeling purposes.

In [ ]:
cat_cols

In [ ]:
num_cols

In [ ]:
cat_but_car

In [ ]:
num_but_cat

In [ ]:
def outlier_thresholds(dataframe, col_name, q1=0.10, q3=0.90):
    quartile1 = dataframe[col_name].quantile(q1)
    quartile3 = dataframe[col_name].quantile(q3)
    interquantile_range = quartile3 - quartile1
    up_limit = quartile3 + 1.5 * interquantile_range
    low_limit = quartile1 - 1.5 * interquantile_range
    return low_limit, up_limit

In [ ]:
def check_outlier(dataframe, col_name):
    low_limit, up_limit = outlier_thresholds(dataframe, col_name)
    if dataframe[(dataframe[col_name] > up_limit) | (dataframe[col_name] < low_limit)].any(axis=None):
        return True
    else:
        return False

In [ ]:
for col in num_cols:
    print(col, check_outlier(df, col))

Only the premium feature contains outliers.

Analysis of Categorical Variables

In [ ]:
def cat_summary(dataframe, col_name, plot=False):
    print(pd.DataFrame({col_name: dataframe[col_name].value_counts(),
                        "Ratio": 100 * dataframe[col_name].value_counts() / len(dataframe)}))
    print("##########################################")
    if plot:
        sns.countplot(x=dataframe[col_name], data=dataframe, palette="viridis")
        plt.xticks(rotation=45)
        plt.show(block=True)

In [ ]:
for col in cat_cols:
    cat_summary(df, col, plot=True)

In [ ]:
# Categorical columns
CatCols = ['gender', 'discount_eligibility', 'region', 'children']
df1[CatCols] = df1[CatCols].astype(str)

# Colors
colors = [
    ['#0055ff', '#00aaff', '#3399ff'],         
    ['#ff7000', '#ffaa00'],                    
    ['#23bf00', '#55dd55'],                   
    ['#cc00cc', '#ff99ff', '#aa00aa', '#dd66dd'] 
]

# Plot
fig, ax = plt.subplots(nrows=1, ncols=4, figsize=(24, 6), dpi=100)

for i, col in enumerate(CatCols):
    chart = sns.countplot(x=col, data=df, ax=ax[i], palette=colors[i])
    ax[i].set_title(f'{col} Distribution', fontsize=18, pad=15)
    ax[i].set_xlabel(col, fontsize=14, labelpad=10)
    ax[i].set_ylabel('Count', fontsize=14, labelpad=10)
    ax[i].tick_params(axis='x', labelsize=13)
    ax[i].tick_params(axis='y', labelsize=12)

    # Add value labels on top of each bar
    for container in chart.containers:
        chart.bar_label(container, fontsize=14, padding=4)

# Main title with space
plt.suptitle('Frequency Distribution of Categorical Variables', fontsize=22, fontweight='bold', y=1.08)
plt.tight_layout()
plt.show()

 Analysis of Numerical Variables

In [ ]:
def num_summary(dataframe, col_name, plot=False):
    quantiles = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.99]
    print(dataframe[col_name].describe(quantiles).T)

    if plot:
        sns.histplot(data=dataframe, x=col_name, bins=20, kde=True, color="blue")
        plt.xticks(rotation=45)
        plt.show(block=True)

In [ ]:
for col in num_cols:
    num_summary(df, col, plot=True)

In [ ]:
# list of 6 columns you want to chart
# colors list
colors = ['skyblue', 'lightgreen', 'salmon', 'plum']
# numeric values 
columns = ['age', 'bmi', 'expenses', 'premium']

# making a large figure with rhe right size 
plt.figure(figsize=(16, 12))

# loop for drawing each chart
for i, col in enumerate(columns):
    plt.subplot(4, 2, i + 1) 
    plt.hist(df1[col], bins=20, color=colors[i%len(colors)], edgecolor='black')
    plt.title(f"{col} Distribution",fontsize=18)

# adjusting distancesا
plt.tight_layout()
plt.show()

In [ ]:

# Select numeric columns to plot
NumCols = ['age', 'bmi', 'expenses', 'premium']

# Setup subplots
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(18, 10), dpi=100)
colors = sns.color_palette("husl", len(NumCols))

for i in range(len(NumCols)):
    row = i // 3
    col = i % 3
    values, bin_edges = np.histogram(df[NumCols[i]], 
                                     range=(np.floor(df[NumCols[i]].min()), np.ceil(df[NumCols[i]].max())))
    
    graph = sns.histplot(data=df, x=NumCols[i], bins=bin_edges, kde=True, ax=ax[row, col],
                         edgecolor='none', color=colors[i], alpha=0.4, line_kws={'lw': 2.5})
    
    ax[row, col].set_xlabel(NumCols[i], fontsize=15)
    ax[row, col].set_ylabel('Count', fontsize=12)
    ax[row, col].set_xticks(np.round(bin_edges, 1))
    ax[row, col].set_xticklabels(ax[row, col].get_xticks(), rotation=45)
    ax[row, col].grid(color='lightgrey')

    for p in graph.patches:
        height = p.get_height()
        if height > 0:
            ax[row, col].annotate(f'{int(height)}',
                                  (p.get_x() + p.get_width() / 2, height + 0.5),
                                  ha='center', fontsize=10, fontweight="bold")

    stats_text = '\n'.join([
        f'μ = {df[NumCols[i]].mean():.2f}',
        f'σ = {df[NumCols[i]].std():.2f}',
        f'Median = {np.median(df[NumCols[i]]):.2f}',
        f'Min = {df[NumCols[i]].min():.2f}',
        f'Max = {df[NumCols[i]].max():.2f}'
    ])

    ax[row, col].text(0.65, 0.95, stats_text, transform=ax[row, col].transAxes,
                      fontsize=10, verticalalignment='top',
                      bbox=dict(boxstyle='round', facecolor=colors[i], edgecolor='black', pad=0.5))

# Turn off the 6th empty plot
ax[1, 2].axis('off')

plt.suptitle('Distribution of Numerical Variables', fontsize=22)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

-1️⃣ Feature: age

 • Mean (μ): 39.21
 • Standard Deviation (σ): 14.05
 • Median: 39
 • Min / Max: 18 / 64
 • Distribution: Fairly spread out with peaks around ages 18 and 50

✅ Interpretation:

 • The age data is well-distributed between 18 and 64.
 • The highest concentration is at age 18 (222 individuals), possibly indicating a large group of young adults.
 • The distribution is not perfectly normal, but it is usable as-is for modeling.

⸻

2️⃣ Feature: expenses

 • Mean (μ): 13,270.42
 • Standard Deviation (σ): 12,110.01
 • Median: 9,382.03
 • Min / Max: 1,121.87 / 63,770.43
 • Distribution: Strong right skew (positively skewed)

✅ Interpretation:

 • The data contains many high-cost outliers.
 • There’s a large gap between the mean and median, indicating heavy skewness.
 • Should be log-transformed or normalized before modeling.

⸻

3️⃣ Feature: premium

 • Mean (μ): 262.87
 • Standard Deviation (σ): 292.53
 • Median: 175
 • Min / Max: 11.22 / 1983.11
 • Distribution: Highly right-skewed

✅ Interpretation:

 • Most customers pay low premiums, with a few paying very high amounts.
 • Outliers significantly affect the mean and spread.
 • Similar to expenses, it may require transformation.

⸻

4️⃣ Feature: bmi

 • Mean (μ): 30.67
 • Standard Deviation (σ): 6.10
 • Median: 30.40
 • Min / Max: 16.00 / 53.10
 • Distribution: Roughly normal (bell-shaped)

✅ Interpretation:

 • The most suitable numerical variable for modeling without preprocessing.
 • Peak occurs in the overweight BMI range (around 30–35).
 • No major skew or outliers.

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;"> Analysis of Categorical Variables by Target</p>

In [ ]:
def target_summary_with_cat(dataframe, target, categorical_col, plot=False):
    print(pd.DataFrame({'TARGET_MEAN': dataframe.groupby(categorical_col)[target].mean()}), end='\n\n\n')
    if plot:
        sns.barplot(x=categorical_col, y=target, data=dataframe, palette="viridis")
        plt.xticks(rotation=45)
        plt.show(block=True)

In [ ]:
for col in cat_cols:
    target_summary_with_cat(df, "expenses", col, plot=True)

In [ ]:
fig, axes = plt.subplots(nrows=1 ,ncols=4, figsize=(12,5), dpi=100)
cat_features = ['gender', 'discount_eligibility', 'region', 'children']
target = 'expenses'
c = '#ffaa00'

for i in range(len(cat_features)):
    sns.stripplot(ax=axes[i], x=cat_features[i], y=target, data=df, size=6, color=c)
    axes[i].set_title('{} vs. {}'.format(target, cat_features[i]), size = 13)
    axes[i].set_xlabel(cat_features[i], size = 12)
    axes[i].set_ylabel(target, size = 12)
    axes[i].grid()

plt.suptitle('expenses vs. Categorical Features', size = 20)
plt.tight_layout()
plt.show()

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Analysis of Numeric Variables by Target</p>

In [ ]:
fig, ax = plt.subplots(nrows=2 ,ncols=2, figsize=(10,10), dpi=90)
num_features = ['age', 'bmi', 'expenses', 'premium']
target = 'expenses'
c = '#0055ff'

for i in range(len(num_features)):
    row = i//2
    col = i%2
    ax[row,col].scatter(df[num_features[i]], df[target], color=c, edgecolors='w', linewidths=0.25)
    ax[row,col].set_title('{} vs. {}'.format(target, num_features[i]), size = 12)
    ax[row,col].set_xlabel(num_features[i], size = 12)
    ax[row,col].set_ylabel(target, size = 12)
    ax[row,col].grid()

plt.suptitle('expenses vs. Numerical Features', size = 20)
plt.tight_layout()
plt.show()

In [ ]:
def target_summary_with_num(dataframe, target, numerical_col, plot=False):
    print(pd.DataFrame({numerical_col+'_mean': dataframe.groupby(target)[numerical_col].mean()}), end='\n\n\n')
    if plot:
        sns.barplot(x=target, y=numerical_col, data=dataframe)
        plt.show(block=True)

In [ ]:
for col in num_cols:
    target_summary_with_num(df, "expenses", col, plot=False)

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
# List of categorical variables to test against the target
categorical_cols = ['gender', 'discount_eligibility', 'region', 'children']
target = 'expenses'

# Loop through each categorical feature
for col in categorical_cols:
    print(f"\n🔹 ANOVA for '{col}' on '{target}':\n")

    # Build an Ordinary Least Squares (OLS) model
    model = ols(f'{target} ~ C({col})', data=df).fit()

    # Perform ANOVA on the fitted model
    anova_table = sm.stats.anova_lm(model, typ=2)

    # Print the ANOVA summary table
    print(anova_table)

    # Extract the p-value from the table
    p_value = anova_table['PR(>F)'][0]

    # Interpret the result: check statistical significance
    if p_value < 0.05:
        print(f"✅ The effect of '{col}' on '{target}' is statistically significant (p-value = {p_value:.4f})")
    else:
        print(f"❌ No statistically significant effect of '{col}' on '{target}' (p-value = {p_value:.4f})")

In [ ]:
# Identify numeric and categorical columns
numerical_cols = ['age', 'bmi', 'expenses', 'premium']
categorical_cols = ['gender', 'discount_eligibility', 'region', 'children']

# -----------------------------
# 1️⃣ NUMERICAL OUTLIERS (IQR)
# -----------------------------

print("🔍 Checking Outliers in Numeric Columns:")
for col in numerical_cols:
    Q1 = df1[col].quantile(0.25)
    Q3 = df1[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df1[(df1[col] < lower) | (df1[col] > upper)]
    percentage = (len(outliers) / len(df1)) * 100
    
    print(f"\n📊 {col}")
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Lower Bound: {lower:.2f}, Upper Bound: {upper:.2f}")
    print(f"  Outliers: {len(outliers)} rows ({percentage:.2f}%)")

    # Boxplot
    plt.figure(figsize=(6, 1.5))
    sns.boxplot(x=df[col], color='blue')
    plt.title(f'Boxplot for {col}')
    plt.tight_layout()
    plt.show()


    

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;"> Correlation Analysis</p>

In [ ]:
def high_correlated_cols(dataframe, plot=False, corr_th=0.70):
    # Only numerical columns
    numeric_df = dataframe.select_dtypes(include=[np.number])

    # Correlation matrix
    corr = numeric_df.corr()
    cor_matrix = corr.abs()

    # Select upper triangular matrix (to remove unnecessary repetitions)
    upper_triangle_matrix = cor_matrix.where(np.triu(np.ones(cor_matrix.shape), k=1).astype(bool))
    # Let's select the column pairs whose correlation is higher than the threshold
    high_corrs = [(col, row)
                  for col in cor_matrix.columns
                  for row in cor_matrix.index
                  if not pd.isna(upper_triangle_matrix.loc[row, col]) and abs(
            upper_triangle_matrix.loc[row, col]) > corr_th]

    if plot:
        # Heatmap
        plt.figure(figsize=(10, 6))
        sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu")
        plt.title("Correlation Heat Map",color="purple")
        plt.show()

    return high_corrs

In [ ]:
high_correlated_cols(df, plot=True)

In [ ]:
selected_cols = num_cols + ['expenses']
selected_df = df[selected_cols]
corr_matrix = selected_df.corr()

In [ ]:
corr_matrix

In [ ]:

# corr just with Selling_Price
expenses_corr = df.corr(numeric_only=True)['expenses'].sort_values(ascending=False)
print(expenses_corr)


<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Distribution of the Dependent Variable</p>

In [ ]:
df["expenses"].hist(bins=100)
plt.show(block=True)

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;"> Examining the Logarithm of the Dependent Variable
</p>

In [ ]:
np.log1p(df['expenses']).hist(bins=50)
plt.show(block=True)

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- • Original (expenses):

 • Highly right-skewed distribution.

 • Most values are concentrated below 10,000.

 • Many outliers on the higher end (up to ~63,000).

 • Not ideal for modeling due to skewness and variance.

 • Log-transformed (np.log1p(expenses)):

 • Distribution becomes much more symmetric and closer to normal.

 • Outliers are compressed, reducing their influence.
 
 • Better suited for regression and machine learning models.

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Missing Value Analysis</p>

In [ ]:
def missing_values_table(dataframe, na_name=False):
    na_columns = [col for col in dataframe.columns if dataframe[col].isnull().sum() > 0]

    n_miss = dataframe[na_columns].isnull().sum().sort_values(ascending=False)

    ratio = (dataframe[na_columns].isnull().sum() / dataframe.shape[0] * 100).sort_values(ascending=False)

    missing_df = pd.concat([n_miss, np.round(ratio, 2)], axis=1, keys=['n_miss', 'ratio'])

    print(missing_df, end="\n")

    if na_name:
        return na_columns

In [ ]:
missing_values_table(df)

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- No columns contain missing values

In [ ]:
plt.figure(1 , figsize=(20 , 35))
n = 0 

for x in ['age', 'bmi', 'expenses', 'premium']:
    
    n += 1
    plt.subplot(9, 2, n)
    plt.subplots_adjust(hspace=0.5, wspace=0.5)

   
    plt.scatter(df[x], df['expenses'], c=df[x], cmap='viridis', alpha=0.8)

    plt.title(f'expenses VS {x}', fontsize=10)
    plt.xlabel(x)
    plt.ylabel('BALANCE')

plt.tight_layout()
plt.show()

In [ ]:
palette ="rocket"
columns1=['age', 'bmi', 'expenses', 'premium','gender', 'discount_eligibility', 'region', 'children']
for column in columns1:
    plt.figure(figsize=(15,2))
    sns.violinplot(x=df[column], palette=palette)
    plt.title(column)
    plt.show()

In [ ]:
df_c=['age', 'bmi', 'expenses', 'premium']  
for i in df_c:
    sns.set_theme(style="white")
    g = sns.JointGrid(data=df, x=i,y='expenses',space=0)
    g.plot_joint(sns.kdeplot,fill=True,thresh=0, levels=100, cmap="viridis")
    g.plot_marginals(sns.histplot, color="#6f4518", alpha=1, bins=20)

In [ ]:
def rare_analyser(dataframe, target, cat_cols):
    for col in cat_cols:
        print(col, ':', len(dataframe[col].value_counts()))
        print(pd.DataFrame({'COUNT': dataframe[col].value_counts(),
                            'RATIO': dataframe[col].value_counts() / len(dataframe),
                            'TARGET_MEAN': dataframe.groupby(col)[target].mean()}), end='\n\n\n')

In [ ]:
rare_analyser(df, "expenses", cat_cols)

In [ ]:
def rare_encoder(dataframe, rare_perc):
    temp_df = dataframe.copy()

    rare_columns = [col for col in temp_df.columns if temp_df[col].dtypes == 'O'
                    and (temp_df[col].value_counts() / len(temp_df) < rare_perc).any(axis=None)]

    for var in rare_columns:
        tmp = temp_df[var].value_counts() / len(temp_df)
        rare_labels = tmp[tmp < rare_perc].index
        temp_df[var] = np.where(temp_df[var].isin(rare_labels), 'Rare', temp_df[var])
    return temp_df

In [ ]:
rare_encoder(df, 0.01)

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- The rare_encoder() function:

• Finds columns that are text (objects).

• If there are values ​​in those columns that are very rarely repeated (e.g. less than 1%), it converts 

those values ​​to 'Rare'.

• Used for simplification before modeling.

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Feature Extraction</p>

In [ ]:
# 1. first male
male_condition = df['gender'] == 'male'
df.loc[male_condition, 'AGE_GROUP'] = pd.cut(
    df.loc[male_condition, 'age'],
    bins=[0, 40, 60, 100],
    labels=["Young_Adults_Male", "Middle-Aged_Adults_Male", "Old_Adults_Male"],
    right=False
)

# 2. Categoric to str
df['AGE_GROUP'] = df['AGE_GROUP'].astype(str)

# 3. Fill NaN with woman
female_condition = df['gender'] == 'female'
df.loc[female_condition, 'AGE_GROUP'] = pd.cut(
    df.loc[female_condition, 'age'],
    bins=[0, 40, 60, 100],
    labels=["Young_Adults_Female", "Middle-Aged_Adults_Female", "Old_Adults_Female"],
    right=False
).astype(str)  # to str

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- • Helps in grouped analysis (e.g., average insurance cost by age group & gender).

 • Makes it easier to build features for machine learning models.
 
 • Improves interpretability of data and visualizations.

In [ ]:
df['NEW_BMI'] = pd.cut(x=df['bmi'], bins=[0, 18.5, 24.9, 29.9, 100],
                       labels=["Underweight", "Healthy", "Overweight", "Obese",]).astype(str)

In [ ]:
df["NEW_AGE_BMI_NOM"] = df["NEW_BMI"] + df["AGE_GROUP"]

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- Creates a new combined categorical feature

In [ ]:
# 1. RISKY: Those who both smoke and are overweight/obese
df['risk_high'] = ((df['discount_eligibility'] == "yes") &
              (df['NEW_BMI'].isin(['Overweight', 'Obese_1', 'Obese_2', 'Obese_3']))).astype(int)

# 2. RISK: Only overweight/obese (including non-smokers)
df['risk_medium'] = df['NEW_BMI'].isin(['Overweight', 'Obese_1', 'Obese_2', 'Obese_3']).astype(int)

In [ ]:
df["risk_score"] = df["risk_high"] * 50 + df["risk_medium"] * 25 + df["age"]

In [ ]:
df['risk_group'] = pd.qcut(x=df['risk_score'], q=4,
                       labels=["no_risk", "medium_risky", "high_risky", "very_high_risky"]).astype(str)

In [ ]:
df['bmi_age_ratio'] = df['bmi'] / df['age']
df['children_discount'] = df['children'] * df['discount_eligibility'].map({'yes': 1, 'no': 0})

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- Is there a connection between having children and getting a discount?

In [ ]:
df['age_squared'] = df['age'] ** 2
df['bmi_cubed'] = df['bmi'] ** 3

In [ ]:
df.columns = [col.upper() for col in df.columns]

In [ ]:
df1=df.head()
df1

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;"> Encoding</p>

In [ ]:
cat_cols, num_cols, cat_but_car,  num_but_cat = grab_col_names(df)

In [ ]:
cat_cols


In [ ]:
num_cols

In [ ]:
cat_but_car

In [ ]:
num_but_cat

In [ ]:
cat_cols = cat_cols + cat_but_car

In [ ]:
cat_cols

In [ ]:
# Label Encoding

def label_encoder(dataframe, binary_col):
    labelencoder = LabelEncoder()
    dataframe[binary_col] = labelencoder.fit_transform(dataframe[binary_col])
    return dataframe

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

- This method is only suitable for binary columns

In [ ]:
binary_cols = [col for col in df.columns if df[col].dtypes == "O" and df[col].nunique() == 2]

In [ ]:
binary_cols


In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
for col in binary_cols:
    df = label_encoder(df, col)

In [ ]:
df.head()

In [ ]:
# One-Hot Encoding

cat_cols = [col for col in cat_cols if col not in binary_cols]

In [ ]:
def one_hot_encoder(dataframe, categorical_cols, drop_first=False):
    dataframe = pd.get_dummies(dataframe, columns=categorical_cols, drop_first=drop_first)
    return dataframe

In [ ]:
df = one_hot_encoder(df, cat_cols)

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
#bool_cols = df.select_dtypes("bool").columns
#df[bool_cols] = df[bool_cols].astype(int)

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;"> Standardization Process</p>

In [ ]:
num_cols = [col for col in num_cols if col not in ["expenses"]]

In [ ]:
from sklearn.preprocessing import RobustScaler

In [ ]:
scaler = RobustScaler()

In [ ]:
df[num_cols] = scaler.fit_transform(df[num_cols])

In [ ]:
df.head(10)

In [ ]:
df.columns

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Creating Model</p>

In [ ]:

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
#from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score,GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder, StandardScaler, RobustScaler

import warnings
warnings.simplefilter(action="ignore")

In [ ]:
df['LOG_EXPENSES'] = np.log1p(df['EXPENSES'])

In [ ]:
y = df["EXPENSES"]

In [ ]:
X = df.drop(["LOG_EXPENSES", "EXPENSES"], axis=1)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
from xgboost import XGBRegressor
import time

In [ ]:
# Model list
models = [('LR', LinearRegression()),
          ("Ridge", Ridge()),
          ("Lasso", Lasso()),
          ("ElasticNet", ElasticNet()),
          ('KNN', KNeighborsRegressor(n_jobs=1)),
          ('CART', DecisionTreeRegressor()),
          ('RF', RandomForestRegressor(n_jobs=1)),
          ('GBM', GradientBoostingRegressor()),
          ("XGBoost", XGBRegressor(objective='reg:squarederror', n_jobs=1)),
          ("CatBoost", CatBoostRegressor(verbose=False, thread_count=1))]
# Initialize empty lists to store metrics and execution times
rmse_scores = []
r2_scores = []
mae_scores = []
mse_scores = []
execution_times = []
model_names = []

# Main loop to fit, predict, and evaluate each model
for name, regressor in models:
    start_time = time.time()

    # Fit the model
    regressor.fit(X_train, y_train)

    # Make predictions
    y_pred = regressor.predict(X_test)

    # Calculate RMSE
    rmse = np.mean(np.sqrt(-cross_val_score(regressor, X, y, cv=5, scoring="neg_mean_squared_error")))
    rmse_scores.append(rmse)
    
    # Calculate R^2 score
    r2 = metrics.r2_score(y_test, y_pred)
    r2_scores.append(r2)

    # Calculate MAE
    mae = metrics.mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    # Calculate MSE
    mse = metrics.mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)

    # Calculate the execution time of the model
    execution_time = time.time() - start_time
    execution_times.append(execution_time)

    # Store model name
    model_names.append(name)

# Create a DataFrame to hold all the metrics and execution times
results_df = pd.DataFrame({
    'Model': model_names,
    'RMSE': rmse_scores,
    'R^2 Score': r2_scores,
    'MAE': mae_scores,
    'MSE': mse_scores,
    'Execution Time (s)': execution_times
})

results_df


<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;"> Hyperparameter optimization</p>

In [ ]:
# Initialize lists to store metrics
rmse_scores = []
r2_scores = []
mae_scores = []
mse_scores = []
execution_times = []
model_names = []

# Define the hyperparameters for each model

param_grids = {
    'LR': {},
    'Ridge': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    'Lasso': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0]},
    'ElasticNet': {'alpha': [0.01, 0.1, 1.0, 10.0, 100.0], 'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]},
    'KNN': {'n_neighbors': [1, 3, 5, 7, 9]},
    'CART': {'max_depth': [None, 5, 10, 20, 30], 'min_samples_leaf': [1, 2, 3]},
    'RF': {'n_estimators': [10, 30, 50, 70, 100], 'max_depth': [None, 5, 10, 20]},
    'GBM': {'n_estimators': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1]},
    'XGBoost': {'n_estimators': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1]},
    # 'LightGBM': {'n_estimators': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1]},
    'CatBoost': {'iterations': [10, 30, 50, 70, 100], 'learning_rate': [0.005, 0.01, 0.05, 0.1], 'depth': [3, 4, 5, 6, 7]}
}

# Variables to store best model and score globally
best_global_score = np.inf
best_global_model = None

# Main loop to fit, predict, and evaluate each model
for name, regressor in models:
    start_time = time.time()
    
    # Hyperparameter tuning with GridSearchCV if param_grids is defined
    if param_grids.get(name):
        grid_search = GridSearchCV(regressor, param_grid=param_grids[name], cv=5, n_jobs=-1)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
    else:
        best_model = regressor
        best_model.fit(X_train, y_train)

    # Make predictions
    y_pred = best_model.predict(X_test)

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    rmse_scores.append(rmse)

    # Update best global model and score
    if rmse < best_global_score:
        best_global_score = rmse
        best_global_model = best_model

    # Calculate other metrics
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

    mse = mean_squared_error(y_test, y_pred)
    mse_scores.append(mse)

    # Calculate execution time
    execution_time = time.time() - start_time
    execution_times.append(execution_time)

    # Store model name
    model_names.append(name)

# Create a DataFrame to store all the metrics and execution times
results_with_tuning_df = pd.DataFrame({
    'Model': model_names,
    'RMSE': rmse_scores,
    'R^2 Score': r2_scores,
    'MAE': mae_scores,
    'MSE': mse_scores,
    'Execution Time (s)': execution_times
})

results_with_tuning_df

In [ ]:
best_global_model

In [ ]:
# Final Prediction Model
final_model = best_global_model

# Make predictions on the test set using the final model
y_final_pred = final_model.predict(X_test)
final_y_pred = (y_final_pred)
final_y_test =(y_test)

In [ ]:
# Creating DataFrame with predicted and actual values
results = pd.DataFrame({'Predicted Value': final_y_pred, 'True Value': final_y_test})

# Adding the difference between actual and forecasted prices as a new column
results['Difference'] = results['True Value'] - results['Predicted Value']

# Add an absolute difference column
results['Absolute Difference'] = results['Difference'].abs()

# Show the 5 worst predictions (largest to smallest by absolute difference)
print("5 worst predictions:")
print(results.sort_values(by='Absolute Difference', ascending=False).head().drop('Absolute Difference', axis=1))

# Show top 5 predictions (smallest to largest by absolute difference)
print("\ntop 5 predictions:")
print(results.sort_values(by='Absolute Difference').head().drop('Absolute Difference', axis=1))

<div style="border-radius:10px; border:#778da9 solid; padding: 15px; background-color:#e0e1dd
            ; font-size:110%; text-align:left">

-  Final Model Predictions and Comparison with True Prices

In [ ]:
# Training the model
best_global_model.fit(X_train, y_train)

# Predictions
y_final_pred = best_global_model.predict(X_test)

# 1. Actual vs. Forecast Scatter Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_final_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel('Real Values')
plt.ylabel('Estimated Values')
plt.title('Actual vs. Forecast Distribution')
plt.show()

# 2. Error Distribution Graph (Residual Plot)
residuals = y_test - y_final_pred
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_final_pred, y=residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Estimated Values')
plt.ylabel('Errors (Actual - Estimated)')
plt.title('Error Distribution')
plt.show()

# 3. Error Density Graph
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True)
plt.xlabel('Error Amount')
plt.ylabel('Frequency')
plt.title('Distribution of Errors')
plt.show()

# 4. Comparison of Actual and Estimated Values (First 100 Examples)
plt.figure(figsize=(12, 6))
plt.plot(y_test.values[:100], label='Real Values', marker='o')
plt.plot(y_final_pred[:100], label='Predictions', marker='x')
plt.xlabel('Sample Index')
plt.ylabel('Value')
plt.title('Comparison of Actual and Estimated Values')
plt.legend()
plt.show()

# Performance Metrics
print("Regression Model Performance Metrics:")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_final_pred)):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_final_pred):.4f}")
print(f"R² Skoru: {r2_score(y_test, y_final_pred):.4f}")

<a id="ml"></a>
# <p style="background-color:#ff69b4;font-family:newtimeroman;font-size:100%;color:black;text-align:center;border-radius:15px 50px; padding:7px;border: 1px solid black;">Feature Importance</p>

In [ ]:
def plot_importance(model, features, num=50, save=False):
    feature_imp = pd.DataFrame({'Value': model.feature_importances_, 'Feature': features.columns})
    plt.figure(figsize=(10, 10))
    sns.set(font_scale=1)
    sns.barplot(x="Value", y="Feature", data=feature_imp.sort_values(by="Value",
                                                                     ascending=False)[0:num])
    plt.title('Features')
    plt.tight_layout()
    plt.show(block=True)
    if save:
        plt.savefig('importances.png')

plot_importance(final_model, X)